# Chapter 9 Walkthrough: File I/O, CSV/JSON Parsing, and Persistent Data

**Book Reference:** Chapter 9, Sections 9.1 – 9.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every demo uses `tempfile.TemporaryDirectory()` so zero stray files are left in your workspace.

## Section 9.3.1 — Modern Paths (`pathlib.Path`) & Safe Text I/O (`with open`)

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    log_path = Path(tmp) / "clinic_shift.txt"
    entries = [
        "08:15 | Dhaka-North | Temp 38.4°C",
        "09:00 | Sylhet-East | Hb 10.6 g/dL",
        "10:30 | Khulna-West | Routine ANC",
    ]
    with open(log_path, "w", encoding="utf-8") as out_f:
        for entry in entries:
            out_f.write(f"{entry}\n")

    print("File name:", log_path.name, "| Exists:", log_path.exists())
    with open(log_path, "r", encoding="utf-8") as in_f:
        for line_no, line in enumerate(in_f, start=1):
            print(f"Line {line_no}: {line.strip()}")


## Section 9.3.2 — Structured Tables with `csv.DictWriter` & `csv.DictReader`

In [ ]:
import csv
from pathlib import Path
import tempfile

records = [
    {"patient_id": "P001", "district": "Dhaka", "hb_g_dl": "10.4"},
    {"patient_id": "P002", "district": "Sylhet", "hb_g_dl": "12.1"},
    {"patient_id": "P003", "district": "Khulna", "hb_g_dl": "9.8"},
]

with tempfile.TemporaryDirectory() as tmp:
    csv_path = Path(tmp) / "mch_cohort.csv"
    fields = ["patient_id", "district", "hb_g_dl"]

    with open(csv_path, "w", encoding="utf-8", newline="") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)

    with open(csv_path, "r", encoding="utf-8", newline="") as in_f:
        loaded = list(csv.DictReader(in_f))

    hb_vals = [float(row["hb_g_dl"]) for row in loaded]
    print("Rows loaded:", len(loaded))
    print("First row:", loaded[0])
    print("Mean Hb (g/dL):", round(sum(hb_vals) / len(hb_vals), 2))


## Section 9.3.3 — Persisting Nested Metadata with `json.dump` & `json.load`

In [ ]:
import json
from pathlib import Path
import tempfile

study_meta = {
    "study_id": "MCH-BD-2026",
    "sites": ["Dhaka", "Sylhet", "Khulna"],
    "cutoffs": {"anemia_g_dl": 11.0, "fever_c": 38.0},
    "active": True,
}

with tempfile.TemporaryDirectory() as tmp:
    json_path = Path(tmp) / "study_meta.json"
    with open(json_path, "w", encoding="utf-8") as out_f:
        json.dump(study_meta, out_f, indent=2)

    with open(json_path, "r", encoding="utf-8") as in_f:
        restored = json.load(in_f)

    print("Study ID:", restored["study_id"])
    print("Anemia cutoff:", restored["cutoffs"]["anemia_g_dl"])
    print("Protocol version:", restored.get("version", "v1.0"))


## Section 9.3.4 — Break It On Purpose! (File I/O & JSON Error Anatomy)

In [ ]:
# Intentional Bug #1: FileNotFoundError — reading a missing CSV file
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    missing_path = Path(tmp) / "nonexistent_cohort.csv"
    with open(missing_path, "r", encoding="utf-8") as f:
        data = f.read()


In [ ]:
# Intentional Bug #2: json.JSONDecodeError — malformed JSON syntax
import json
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    broken_path = Path(tmp) / "broken_report.json"
    broken_path.write_text('{"Dengue": 14, "Cholera": }', encoding="utf-8")
    with open(broken_path, "r", encoding="utf-8") as f:
        report = json.load(f)


In [ ]:
# Intentional Bug #3: io.UnsupportedOperation — writing to read-only file
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    ro_path = Path(tmp) / "readonly_log.txt"
    ro_path.write_text("Existing shift entry\n", encoding="utf-8")
    with open(ro_path, "r", encoding="utf-8") as f:
        f.write("New urgent entry\n")


## Section 9.4 — Worked Problem-Structuring Blueprint: Messy CSV to District JSON Report

In [ ]:
import csv
import json
from pathlib import Path
import tempfile

def csv_to_district_hb_json(in_csv, out_json):
    """Aggregate valid Hb by normalized district and save JSON."""
    grouped = {}
    with open(in_csv, "r", encoding="utf-8", newline="") as in_f:
        for row in csv.DictReader(in_f):
            raw_hb = row["hb_g_dl"].strip()
            if raw_hb in ("", "N/A", "999"):
                continue
            district = row["district"].strip().title()
            grouped.setdefault(district, []).append(float(raw_hb))

    summary = {
        dist: round(sum(vals) / len(vals), 2)
        for dist, vals in grouped.items()
    }
    with open(out_json, "w", encoding="utf-8") as out_f:
        json.dump(summary, out_f, indent=2)
    return summary

with tempfile.TemporaryDirectory() as tmp:
    raw_csv = Path(tmp) / "raw_hb.csv"
    out_json = Path(tmp) / "district_hb.json"
    raw_csv.write_text(
        "id,district,hb_g_dl\n"
        "M1,  dhaka ,10.4\n"
        "M2,DHAKA,999\n"
        "M3,Dhaka,11.6\n"
        "M4,sylhet,12.2\n"
        "M5,Sylhet,N/A\n"
        "M6,SYLHET,11.4\n",
        encoding="utf-8",
    )
    report = csv_to_district_hb_json(raw_csv, out_json)
    print("Returned summary:", report)
    print("Saved JSON:\n" + out_json.read_text(encoding="utf-8"))
